In [ ]:
!pip install python-docx keybert bertopic sentence-transformers transformers scikit-learn tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 15.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files
files.upload()


Saving OS-20251228T101137Z-1-001.zip to OS-20251228T101137Z-1-001.zip


{'OS-20251228T101137Z-1-001.zip': b'PK\x03\x04\x14\x00\x08\x08\x08\x00\xa4!\x99[\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x10\x00\x19\x00OS/DeadLock.docxup\x15\x00\x01\x84\xc2\x98\xbcOS/DeadLock.docx\x01\xaa^U\xa1PK\x03\x04\x14\x00\x08\x08\x08\x00\xce!\x99[\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x12\x00\x00\x00word/numbering.xml\xa5\x94Kn\xc20\x10\x86O\xd0; \xef!\x01\xb5\x88F\x04\x16E\xed\xa6\xbb\xb6\x07\x18l\x93X\xf8\xa5\xb1\xc3\xe3\xf6u \x0f\nU\x15\xd2U\x12\x8f\xffo\xc6\x93\xdf3_\x1e\x94\x1c\xec8:atJ\xc6\xa3\x98\x0c\xb8\xa6\x86\t\x9d\xa5\xe4\xeb\xf3u8#\x03\xe7A3\x90F\xf3\x94\x1c\xb9#\xcb\xc5\xc3|\x9f\xe8B\xad9\x86}\x83\x80\xd0.Q4%\xb9\xf76\x89"Gs\xae\xc0\x8d\x8c\xe5:\x047\x06\x15\xf8\xf0\x89Y\xa4\x00\xb7\x85\x1dR\xa3,x\xb1\x16R\xf8c4\x89\xe3)\xa90&%\x05\xea\xa4B\x0c\x95\xa0h\x9c\xd9\xf8R\x92\x98\xcdFP^=j\x05v\xc9{\x96\xac\x0c-\x14\xd7\xfe\x941B.C\rF\xbb\\XW\xd3T_Z\x08\xe65d\xf7\xd7!vJ\xd6\xfb\xf6\xb6K6\x86\xb0\x0f}V\xf2\x9cho\x90Y4\x94;\x17VW\xe7`C\x1c\xc7\x1d\x1aX"\x

In [ ]:
!unzip *.zip


Archive:  OS-20251228T101137Z-1-001.zip
  inflating: OS/DeadLock.docx        
  inflating: OS/CPU scheduling.docx  
  inflating: OS/Process Management dataset.docx  
  inflating: OS/Memory Management.docx  
  inflating: OS/Context Switching.docx  


In [ ]:
def chunk_text(text):
    return [p.strip() for p in text.split(".") if len(p.strip()) > 40]


In [ ]:
from docx import Document
import os

def read_all_docx(folder):
    files_data = []
    for f in os.listdir(folder):
        if f.endswith(".docx"):
            doc = Document(os.path.join(folder, f))
            text = " ".join(p.text for p in doc.paragraphs if p.text.strip())
            files_data.append({
                "file": f,
                "text": text
            })
    return files_data
docs = read_all_docx("/content/OS")

documents = []
for d in docs:
    chunks = chunk_text(d["text"])
    for c in chunks:
        documents.append({
            "file": d["file"],
            "text": c
        })


In [ ]:
from keybert import KeyBERT

kw_model = KeyBERT("all-MiniLM-L6-v2")

raw_phrases = []
for item in documents:
    kws = kw_model.extract_keywords(
        item["text"],
        keyphrase_ngram_range=(2, 4),
        stop_words="english",
        top_n=3
    )
    raw_phrases.extend([k[0].lower() for k in kws])



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
BAD_WORDS = [
    "allows", "enables", "defined", "usually", "involves",
    "system", "processes", "resources", "stored"
]

def canonicalize(p):
    for b in BAD_WORDS:
        p = p.replace(b, "")
    return " ".join(p.split())

clean_phrases = set()
for p in raw_phrases:
    p = canonicalize(p)
    if 2 <= len(p.split()) <= 4:
        clean_phrases.add(p)

clean_phrases = list(clean_phrases)
print("Clean phrases:", len(clean_phrases))


Clean phrases: 3681


In [ ]:
phrases = clean_phrases  # use all phrases, no filter



In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 4),
    min_df=2
)

topic_model = BERTopic(
    vectorizer_model=vectorizer,
    language="english"
)

texts_only = [d["text"] for d in documents]
topics, _ = topic_model.fit_transform(texts_only)

topic_keywords = {}
for t in set(topics):
    if t != -1:
        topic_keywords[f"Topic_{t}"] = [
            w for w, _ in topic_model.get_topic(t)
        ]


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
def os_label(entity):
    e = entity.lower()

    if "round robin" in e or "fcfs" in e or "sjf" in e:
        return "Scheduling Algorithm"

    if "preemptive" in e or "non preemptive" in e:
        return "Scheduling Technique"

    if "paging" in e or "segmentation" in e:
        return "Memory Management Technique"

    if "context switching" in e:
        return "Concept"

    if "deadlock" in e:
        return "Concept"

    if "cpu scheduling" in e or "process scheduling" in e:
        return "Topic"

    return "Concept"


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_entity(phrase):
    doc = nlp(phrase)
    noun_chunks = list(doc.noun_chunks)
    if noun_chunks:
        chunk = noun_chunks[-1]  # usually last chunk is entity
        words = [token.lemma_ for token in chunk if not token.is_stop and token.is_alpha]
        return " ".join(words[:3])
    for token in reversed(list(doc)):  # Changed from doc[::-1] to reversed(list(doc))
        if token.pos_ == "NOUN":
            return token.lemma_
    return None

In [ ]:
entity_cache = {}

for phrase in clean_phrases:
    entity = extract_entity(phrase)
    if entity and len(entity.split()) <= 3:
        entity_cache[entity] = os_label(entity)

print("Unique entities extracted:", len(entity_cache))


Unique entities extracted: 2152


In [ ]:
from collections import defaultdict

file_entities = defaultdict(dict)

for item in documents:
    file_name = item["file"]

    for entity, label in entity_cache.items():
        file_entities[file_name][entity] = label


In [ ]:
final_output = []

for file_name, ents in file_entities.items():
    final_output.append({
        "file": file_name,
        "entities": [
            {"entity": e, "label": l}
            for e, l in ents.items()
        ]
    })


In [ ]:
import json

with open("OS-Annotation(combined).json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)


# Task
Process each document in the '/content/OS' folder individually to extract topics and entities, and then output these results in a JSON format, with each document's results keyed by its filename.

## Initialize Output Structure

### Subtask:
Create an empty dictionary to store the processed topics and entities for each document, using the document's filename as the key.


**Reasoning**:
Create an empty dictionary `all_document_results` as specified in the instructions to store the processed topics and entities for each document.



In [ ]:
all_document_results = {}
print("Initialized all_document_results dictionary.")

Initialized all_document_results dictionary.


## Process Each Document Individually

### Subtask:
Iterate through each .docx file in the '/content/OS' folder. For each document, perform the following steps: read its content, chunk the text, extract keywords using KeyBERT, clean and canonicalize these phrases, extract and label entities using spaCy, and apply BERTopic to identify topics and their keywords specifically for that document's content. Store these results in the dictionary created in the previous step, keyed by the document's filename.


**Reasoning**:
The subtask requires iterating through each DOCX file, processing it individually using the previously defined functions and models (chunking, KeyBERT, canonicalization, spaCy entity extraction/labeling, and BERTopic), and storing the results keyed by filename in the `all_document_results` dictionary.



In [ ]:
import os
from docx import Document
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

for filename in os.listdir('/content/OS'):
    if filename.endswith(".docx"):
        doc_path = os.path.join('/content/OS', filename)
        doc_name = os.path.splitext(filename)[0]

        # 1. Read document content
        doc = Document(doc_path)
        text = " ".join(p.text for p in doc.paragraphs if p.text.strip())

        # 2. Chunk the text
        doc_chunks = chunk_text(text)

        # Skip if no chunks are found for this document
        if not doc_chunks:
            print(f"No valid chunks found for {doc_name}. Skipping.")
            continue

        # 3. Extract keywords using KeyBERT
        doc_raw_phrases = []
        for chunk in doc_chunks:
            kws = kw_model.extract_keywords(
                chunk,
                keyphrase_ngram_range=(2, 4),
                stop_words="english",
                top_n=3
            )
            doc_raw_phrases.extend([k[0].lower() for k in kws])

        # 4. Clean and canonicalize phrases
        doc_clean_phrases = set()
        for p in doc_raw_phrases:
            p = canonicalize(p)
            if 2 <= len(p.split()) <= 4:
                doc_clean_phrases.add(p)
        doc_clean_phrases = list(doc_clean_phrases)

        # 5. Extract and label entities using spaCy
        doc_final_entities = {}
        for phrase in doc_clean_phrases:
            entity = extract_entity(phrase)
            if entity and len(entity.split()) <= 3:
                doc_final_entities[entity] = os_label(entity)

        # 6. Apply BERTopic for individual document
        # Re-initialize CountVectorizer and BERTopic for each document
        doc_vectorizer = CountVectorizer(
            stop_words="english",
            ngram_range=(2, 4),
            min_df=2
        )
        doc_topic_model = BERTopic(
            vectorizer_model=doc_vectorizer,
            language="english"
        )

        doc_topics, _ = doc_topic_model.fit_transform(doc_chunks)

        doc_topic_keywords = {}
        for t in set(doc_topics):
            if t != -1:
                doc_topic_keywords[f"Topic_{t}"] = [
                    w for w, _ in doc_topic_model.get_topic(t)
                ]

        # 7. Store results in all_document_results
        all_document_results[doc_name] = {
            "Topics": doc_topic_keywords,
            "Entities": doc_final_entities
        }
        print(f"Finished processing {doc_name}.")

print("All documents processed. Results stored in all_document_results.")


ModuleNotFoundError: No module named 'docx'

**Reasoning**:
The previous code failed because the 'docx' module was not found, despite a prior installation attempt. To ensure the dependency is met before attempting to import, the `!pip install python-docx` command is added at the beginning of the cell.



In [ ]:
import os
from docx import Document
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

!pip install python-docx # Ensure python-docx is installed and available

for filename in os.listdir('/content/OS'):
    if filename.endswith(".docx"):
        doc_path = os.path.join('/content/OS', filename)
        doc_name = os.path.splitext(filename)[0]

        # 1. Read document content
        doc = Document(doc_path)
        text = " ".join(p.text for p in doc.paragraphs if p.text.strip())

        # 2. Chunk the text
        doc_chunks = chunk_text(text)

        # Skip if no chunks are found for this document
        if not doc_chunks:
            print(f"No valid chunks found for {doc_name}. Skipping.")
            continue

        # 3. Extract keywords using KeyBERT
        doc_raw_phrases = []
        for chunk in doc_chunks:
            kws = kw_model.extract_keywords(
                chunk,
                keyphrase_ngram_range=(2, 4),
                stop_words="english",
                top_n=3
            )
            doc_raw_phrases.extend([k[0].lower() for k in kws])

        # 4. Clean and canonicalize phrases
        doc_clean_phrases = set()
        for p in doc_raw_phrases:
            p = canonicalize(p)
            if 2 <= len(p.split()) <= 4:
                doc_clean_phrases.add(p)
        doc_clean_phrases = list(doc_clean_phrases)

        # 5. Extract and label entities using spaCy
        doc_final_entities = {}
        for phrase in doc_clean_phrases:
            entity = extract_entity(phrase)
            if entity and len(entity.split()) <= 3:
                doc_final_entities[entity] = os_label(entity)

        # 6. Apply BERTopic for individual document
        # Re-initialize CountVectorizer and BERTopic for each document
        doc_vectorizer = CountVectorizer(
            stop_words="english",
            ngram_range=(2, 4),
            min_df=2
        )
        doc_topic_model = BERTopic(
            vectorizer_model=doc_vectorizer,
            language="english"
        )

        doc_topics, _ = doc_topic_model.fit_transform(doc_chunks)

        doc_topic_keywords = {}
        for t in set(doc_topics):
            if t != -1:
                doc_topic_keywords[f"Topic_{t}"] = [
                    w for w, _ in doc_topic_model.get_topic(t)
                ]

        # 7. Store results in all_document_results
        all_document_results[doc_name] = {
            "Topics": doc_topic_keywords,
            "Entities": doc_final_entities
        }
        print(f"Finished processing {doc_name}.")

print("All documents processed. Results stored in all_document_results.")


ModuleNotFoundError: No module named 'docx'

**Reasoning**:
The `!pip install` command should ideally be in its own cell to ensure the package is fully installed and available in the Python environment before subsequent `import` statements are executed. Placing it in a separate cell guarantees this execution order, resolving the `ModuleNotFoundError`.



In [ ]:
import os
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

# Ensure python-docx is installed and available
# The import statement for Document is moved below the pip install to ensure it's loaded after installation.
from docx import Document

for filename in os.listdir('/content/OS'):
    if filename.endswith(".docx"):
        doc_path = os.path.join('/content/OS', filename)
        doc_name = os.path.splitext(filename)[0]

        # 1. Read document content
        doc = Document(doc_path)
        text = " ".join(p.text for p in doc.paragraphs if p.text.strip())

        # 2. Chunk the text
        doc_chunks = chunk_text(text)

        # Skip if no chunks are found for this document
        if not doc_chunks:
            print(f"No valid chunks found for {doc_name}. Skipping.")
            continue

        # 3. Extract keywords using KeyBERT
        doc_raw_phrases = []
        for chunk in doc_chunks:
            kws = kw_model.extract_keywords(
                chunk,
                keyphrase_ngram_range=(2, 4),
                stop_words="english",
                top_n=3
            )
            doc_raw_phrases.extend([k[0].lower() for k in kws])

        # 4. Clean and canonicalize phrases
        doc_clean_phrases = set()
        for p in doc_raw_phrases:
            p = canonicalize(p)
            if 2 <= len(p.split()) <= 4:
                doc_clean_phrases.add(p)
        doc_clean_phrases = list(doc_clean_phrases)

        # 5. Extract and label entities using spaCy
        doc_final_entities = {}
        for phrase in doc_clean_phrases:
            entity = extract_entity(phrase)
            if entity and len(entity.split()) <= 3:
                doc_final_entities[entity] = os_label(entity)

        # 6. Apply BERTopic for individual document
        # Re-initialize CountVectorizer and BERTopic for each document
        doc_vectorizer = CountVectorizer(
            stop_words="english",
            ngram_range=(2, 4),
            min_df=2
        )
        doc_topic_model = BERTopic(
            vectorizer_model=doc_vectorizer,
            language="english"
        )

        doc_topics, _ = doc_topic_model.fit_transform(doc_chunks)

        doc_topic_keywords = {}
        for t in set(doc_topics):
            if t != -1:
                doc_topic_keywords[f"Topic_{t}"] = [
                    w for w, _ in doc_topic_model.get_topic(t)
                ]

        # 7. Store results in all_document_results
        all_document_results[doc_name] = {
            "Topics": doc_topic_keywords,
            "Entities": doc_final_entities
        }
        print(f"Finished processing {doc_name}.")

print("All documents processed. Results stored in all_document_results.")


ModuleNotFoundError: No module named 'bertopic'

**Reasoning**:
The `ModuleNotFoundError` for 'bertopic' indicates that the library is not found in the current environment. To ensure all necessary packages, including 'bertopic' and 'python-docx' (which also had a `ModuleNotFoundError` in previous attempts), are available for the current execution, the full installation command should be placed at the very beginning of the code block, before any imports. This ensures that the dependencies are met before they are used.



In [ ]:
import os
import sys
!{sys.executable} -m pip install python-docx keybert bertopic sentence-transformers transformers scikit-learn tqdm
from docx import Document
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

for filename in os.listdir('/content/OS'):
    if filename.endswith(".docx"):
        doc_path = os.path.join('/content/OS', filename)
        doc_name = os.path.splitext(filename)[0]

        # 1. Read document content
        doc = Document(doc_path)
        text = " ".join(p.text for p in doc.paragraphs if p.text.strip())

        # 2. Chunk the text
        doc_chunks = chunk_text(text)

        # Skip if no chunks are found for this document
        if not doc_chunks:
            print(f"No valid chunks found for {doc_name}. Skipping.")
            continue

        # 3. Extract keywords using KeyBERT
        doc_raw_phrases = []
        for chunk in doc_chunks:
            kws = kw_model.extract_keywords(
                chunk,
                keyphrase_ngram_range=(2, 4),
                stop_words="english",
                top_n=3
            )
            doc_raw_phrases.extend([k[0].lower() for k in kws])

        # 4. Clean and canonicalize phrases
        doc_clean_phrases = set()
        for p in doc_raw_phrases:
            p = canonicalize(p)
            if 2 <= len(p.split()) <= 4:
                doc_clean_phrases.add(p)
        doc_clean_phrases = list(doc_clean_phrases)

        # 5. Extract and label entities using spaCy
        doc_final_entities = {}
        for phrase in doc_clean_phrases:
            entity = extract_entity(phrase)
            if entity and len(entity.split()) <= 3:
                doc_final_entities[entity] = os_label(entity)

        # 6. Apply BERTopic for individual document
        # Re-initialize CountVectorizer and BERTopic for each document
        doc_vectorizer = CountVectorizer(
            stop_words="english",
            ngram_range=(2, 4),
            min_df=2
        )
        doc_topic_model = BERTopic(
            vectorizer_model=doc_vectorizer,
            language="english"
        )

        doc_topics, _ = doc_topic_model.fit_transform(doc_chunks)

        doc_topic_keywords = {}
        for t in set(doc_topics):
            if t != -1:
                doc_topic_keywords[f"Topic_{t}"] = [
                    w for w, _ in doc_topic_model.get_topic(t)
                ]

        # 7. Store results in all_document_results
        all_document_results[doc_name] = {
            "Topics": doc_topic_keywords,
            "Entities": doc_final_entities
        }
        print(f"Finished processing {doc_name}.")

print("All documents processed. Results stored in all_document_results.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


FileNotFoundError: [Errno 2] No such file or directory: '/content/OS'

**Reasoning**:
The previous execution failed with a `FileNotFoundError` because the directory '/content/OS' could not be found, despite earlier steps showing successful unzipping to this location. To make the code robust against transient file system issues or kernel resets, I will add a check for the directory's existence before attempting to list its contents. If the directory is missing, an informative error message will be printed. Additionally, the `!pip install` command is removed from this cell as it should be in a separate, dedicated setup cell to ensure dependencies are loaded correctly before execution of the main logic.



In [ ]:
import os
from docx import Document
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

# Ensure that the /content/OS directory exists before proceeding.
# This handles cases where the unzipped files might be temporarily unavailable or the kernel state reset.
if not os.path.exists('/content/OS'):
    print("Error: '/content/OS' directory not found. Please ensure the DOCX files are uploaded and unzipped correctly into this path.")
else:
    for filename in os.listdir('/content/OS'):
        if filename.endswith(".docx"):
            doc_path = os.path.join('/content/OS', filename)
            doc_name = os.path.splitext(filename)[0]

            # 1. Read document content
            doc = Document(doc_path)
            text = " ".join(p.text for p in doc.paragraphs if p.text.strip())

            # 2. Chunk the text
            doc_chunks = chunk_text(text)

            # Skip if no chunks are found for this document
            if not doc_chunks:
                print(f"No valid chunks found for {doc_name}. Skipping.")
                continue

            # 3. Extract keywords using KeyBERT
            doc_raw_phrases = []
            for chunk in doc_chunks:
                kws = kw_model.extract_keywords(
                    chunk,
                    keyphrase_ngram_range=(2, 4),
                    stop_words="english",
                    top_n=3
                )
                doc_raw_phrases.extend([k[0].lower() for k in kws])

            # 4. Clean and canonicalize phrases
            doc_clean_phrases = set()
            for p in doc_raw_phrases:
                p = canonicalize(p)
                if 2 <= len(p.split()) <= 4:
                    doc_clean_phrases.add(p)
            doc_clean_phrases = list(doc_clean_phrases)

            # 5. Extract and label entities using spaCy
            doc_final_entities = {}
            for phrase in doc_clean_phrases:
                entity = extract_entity(phrase)
                if entity and len(entity.split()) <= 3:
                    doc_final_entities[entity] = os_label(entity)

            # 6. Apply BERTopic for individual document
            # Re-initialize CountVectorizer and BERTopic for each document
            doc_vectorizer = CountVectorizer(
                stop_words="english",
                ngram_range=(2, 4),
                min_df=2
            )
            doc_topic_model = BERTopic(
                vectorizer_model=doc_vectorizer,
                language="english"
            )

            doc_topics, _ = doc_topic_model.fit_transform(doc_chunks)

            doc_topic_keywords = {}
            for t in set(doc_topics):
                if t != -1:
                    doc_topic_keywords[f"Topic_{t}"] = [
                        w for w, _ in doc_topic_model.get_topic(t)
                    ]

            # 7. Store results in all_document_results
            all_document_results[doc_name] = {
                "Topics": doc_topic_keywords,
                "Entities": doc_final_entities
            }
            print(f"Finished processing {doc_name}.")

    print("All documents processed. Results stored in all_document_results.")


Error: '/content/OS' directory not found. Please ensure the DOCX files are uploaded and unzipped correctly into this path.


## Process Each Document Individually

### Subtask:
Iterate through each .docx file in the '/content/OS' folder. For each document, perform the following steps: read its content, chunk the text, extract keywords using KeyBERT, clean and canonicalize these phrases, extract and label entities using spaCy, and apply BERTopic to identify topics and their keywords specifically for that document's content. Store these results in the dictionary created in the previous step, keyed by the document's filename.


**Reasoning**:
The previous execution failed because the '/content/OS' directory was not found. This indicates that the zip file containing the documents might not have been unzipped or the environment state was reset. Therefore, the next step is to explicitly unzip the provided zip file to ensure the '/content/OS' directory exists before attempting to process its contents.



In [ ]:
!unzip OS-20251228T101137Z-1-001.zip

unzip:  cannot find or open OS-20251228T101137Z-1-001.zip, OS-20251228T101137Z-1-001.zip.zip or OS-20251228T101137Z-1-001.zip.ZIP.


**Reasoning**:
The previous unzip command failed because the zip file was not found. To resolve this, I need to re-upload the zip file to ensure it's available in the Colab environment before attempting to unzip it again.



In [ ]:
from google.colab import files
files.upload()

KeyboardInterrupt: 

# Task
I understand. Please re-upload the `OS-20251228T101137Z-1-001.zip` file when prompted by the `files.upload()` function in the next cell. This step is crucial for the rest of the process to work.

## Re-upload Zip File

### Subtask:
The previous file upload was interrupted. Please re-upload the 'OS-20251228T101137Z-1-001.zip' file when prompted by the `files.upload()` function in the next code cell. This step is crucial for the rest of the process to work.


**Reasoning**:
The previous file upload was interrupted, so the subtask is to re-upload the zip file. This requires running the `files.upload()` function again and waiting for user interaction.



# Task
Okay, I will proceed with the plan to resolve the `NameError` and correct the document processing logic.

**Plan:**

1.  **Move `chunk_text` function definition**: Move the definition of the `chunk_text` function from cell `IvZXVl8Iiiw0` to cell `M4eu4lmmiWRY` so that it is defined before it is called.
2.  **Fix document processing logic**: Correct the loop in cell `M4eu4lmmiWRY` that processes the `docs` list. Currently, it passes an entire document dictionary (`d`) to `chunk_text` instead of `d["text"]`, and it should use `documents.extend` with the `chunks` directly to avoid nesting.
3.  **Execute corrected cell**: Execute cell `M4eu4lmmiWRY` with the corrected `chunk_text` definition and document processing logic.

I will begin by moving the `chunk_text` function definition.

## Move chunk_text function definition

### Subtask:
Move the definition of the `chunk_text` function from cell `IvZXVl8Iiiw0` to cell `M4eu4lmmiWRY` so that it is defined before it is called.


**Reasoning**:
The user explicitly requested to move the `chunk_text` function definition from cell `IvZXVl8Iiiw0` to cell `M4eu4lmmiWRY`. This will ensure the function is defined before it's called, resolving the `NameError` encountered previously in `M4eu4lmmiWRY`.



In [ ]:
from docx import Document
import os

def read_all_docx(folder):
    files_data = []
    for f in os.listdir(folder):
        if f.endswith(".docx"):
            doc = Document(os.path.join(folder, f))
            text = " ".join(p.text for p in doc.paragraphs if p.text.strip())
            files_data.append({
                "file": f,
                "text": text
            })
    return files_data

def chunk_text(text):
    return [p.strip() for p in text.split(".") if len(p.strip()) > 40]

docs = read_all_docx("/content/OS")

documents = []
for d in docs:
    chunks = chunk_text(d["text"])
    for c in chunks:
        documents.append({
            "file": d["file"],
            "text": c
        })


**Reasoning**:
The `chunk_text` function definition has been moved to cell `d09ea87d` (M4eu4lmmiWRY in the original notebook description), and the document processing logic within that cell has also been corrected. Executing this cell will define the `chunk_text` function and process the initial set of documents, resolving the `NameError` and advancing the overall task.



In [ ]:
from docx import Document
import os

def read_all_docx(folder):
    files_data = []
    for f in os.listdir(folder):
        if f.endswith(".docx"):
            doc = Document(os.path.join(folder, f))
            text = " ".join(p.text for p in doc.paragraphs if p.text.strip())
            files_data.append({
                "file": f,
                "text": text
            })
    return files_data

def chunk_text(text):
    return [p.strip() for p in text.split(".") if len(p.strip()) > 40]

docs = read_all_docx("/content/OS")

documents = []
for d in docs:
    chunks = chunk_text(d["text"])
    for c in chunks:
        documents.append({
            "file": d["file"],
            "text": c
        })


## Re-upload Zip File

### Subtask:
Re-upload the 'OS-20251228T101137Z-1-001.zip' file to the Colab environment.


**Reasoning**:
The subtask explicitly requires re-uploading the zip file. The `files.upload()` command is used to prompt the user to select and upload a file to the Colab environment.



In [ ]:
from google.colab import files
files.upload()

KeyboardInterrupt: 